# TP ChatBot avec Langgraph de Langchain

### 1. Config

In [52]:
%pip install langchain langchain-tavily

In [14]:
%pip install langchain langchain-huggingface huggingface_hub

In [24]:
%pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.3 MB/s eta 0:00:00


In [26]:
import os
from google.colab import userdata

os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

### 2. Modèle

In [27]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7,
    max_tokens=256,
)

response = llm.invoke("Explique LangChain en deux phrases.")
print(response.content)

LangChain est une bibliothèque logicielle open-source qui permet de créer des applications intégrant l'intelligence artificielle, notamment les modèles de langage, pour automatiser des tâches complexes et développer des chatbots, assistants virtuels et systèmes de recommandation. Elle facilite la création de chaines de traitement de textes pour traiter les données, générer du contenu et prendre des décisions basées sur l'analyse de langage naturel.


### 3. Messages

In [29]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage("Tu es un assistant d'aide au code. Tu aides à trouver des librairies/technologies utiles pour les besoins de l'utilisateur.")
human_msg = HumanMessage("Salut, je souhaite travailler sur un site web de cartographie 2D en javascript")

messages = [system_msg, human_msg]
response = llm.invoke(messages)
print(response.content)

Excellente idée ! Il existe plusieurs options pour créer un site web de cartographie 2D en JavaScript. Voici quelques suggestions de librairies et technologies qui pourraient vous aider :

1. **Leaflet** : Leaflet est une des bibliothèques de cartographie les plus populaires pour JavaScript. Il propose une prise en charge de la géolocalisation, de la zoom, de la rotation, de la lecture de couches de données géographiques, etc.
2. **Mapbox** : Mapbox est une plateforme de cartographie qui propose une API JavaScript pour créer des cartes 2D et 3D. Elle offre des fonctionnalités avancées comme la géolocalisation, la prise en charge de la navigation, etc.
3. **OpenLayers** : OpenLayers est une bibliothèque de cartographie open-source qui propose une prise en charge de la géolocalisation, de la zoom, de la rotation, de la lecture de couches de données géographiques, etc.
4. **Google Maps JavaScript API** : La bibliothèque Google Maps JavaScript est une API qui permet de créer des cartes 2D 

In [30]:
# String content
human_message_ex = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message_ex = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message_ex = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

In [31]:
#using ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant d'aide au code. "
        "Tu aides à trouver des librairies et technologies adaptées."
    ),
    (
        "human",
        "Je souhaite travailler sur un projet de type {type_projet} "
        "en utilisant {langage}."
    ),
])

messages = prompt.invoke({
    "type_projet": "site web de cartographie 2D",
    "langage": "JavaScript",
})

response = llm.invoke(messages)
print(response.content)

Excellente idée ! Il existe de nombreuses bibliothèques JavaScript qui peuvent vous aider à créer un site web de cartographie 2D. Voici quelques-unes des options les plus populaires :

1. **Leaflet** : Leaflet est une bibliothèque JavaScript très populaire pour la création de cartes 2D. Elle est légère, flexible et facile à utiliser. Leaflet prend en charge de nombreux formats de cartes (tiles) et permet la personnalisation de la carte (couleurs, symboles, etc.).
2. **OpenLayers** : OpenLayers est une autre bibliothèque JavaScript pour la création de cartes 2D. Elle est conçue pour être utilisée avec des services de cartographie comme Google Maps, Bing Maps ou OpenStreetMap.
3. **D3.js** : D3.js (Data-Driven Documents) est une bibliothèque JavaScript pour la création de visualisations données. Elle peut être utilisée pour créer des cartes 2D personnalisées.
4. **Mapbox GL JS** : Mapbox GL JS est une bibliothèque JavaScript pour la création de cartes 2D et 3D.


### 4. Outil de recherche Tavily

In [39]:
from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=2,
    search_depth="basic",
    include_answer=True,
)

tools = [search]

### 5. Agent

In [53]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    system_prompt=(
        "Tu es un assistant spécialisé dans la recommandation de technologies "
        "et de bibliothèques informatiques.\n\n"

        "L'utilisateur fournit :\n"
        "- un besoin ou un type de projet ;\n"
        "- un langage ou une stack technique.\n\n"

        "Propose entre 3 et 5 solutions adaptées.\n"
        "Utilise l'outil de recherche web pour vérifier la licence, "
        "la tarification et les éventuelles restrictions d'utilisation.\n"
        "Privilégie les sources officielles.\n\n"

        "Retourne uniquement un tableau Markdown avec les colonnes suivantes :\n"
        "| Solution | Description | Licence ou tarification |\n\n"

        "Dans la dernière colonne, précise par exemple : MIT, Apache 2.0, "
        "open source, gratuit, freemium ou payant. "
        "Si l'information n'a pas pu être vérifiée, indique 'À vérifier'."
    ),
)

In [55]:
config = {
    "configurable": {
        "thread_id": "projet-cartographie"
    }
}

### 6. Invocation

In [56]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Type de projet : création et affichage d'assets représentant "
                "des véhicules terrestres sur une carte tactique.\n"
                "Stack actuelle : JavaScript, Leaflet et HTML."
                )
        }
    ]
},

config=config,

)

print(result["messages"][-1].content)

Voici les solutions adaptées pour la création et l'affichage d'assets représentant des véhicules terrestres sur une carte tactique :

| Solution | Description | Licence ou tarification |
| --- | --- | --- |
| Three.js | Bibliothèque JavaScript pour la création et l'affichage de 3-D | Licence MIT |
| ArcGIS API for JavaScript | API de mapping pour la création de scènes 3-D et la traçabilité en temps réel | Tarification payante |
| Leaflet | Bibliothèque JavaScript pour l'affichage de cartes interactives | Licence open source |
| OpenLayers | Bibliothèque JavaScript pour l'affichage de cartes 2-D | Licence open source |
| Mapbox GL JS | Bibliothèque JavaScript pour l'affichage de cartes 2-D avec des couches 3-D | Tarification payante |
| CesiumJS | Bibliothèque JavaScript pour l'affichage de cartes 3-D avec des couches 2-D | Licence open source |

Notez que certaines de ces solutions nécessitent une tarification payante ou une licence spécifique. Il est donc important de vérifier les con

In [57]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Parmi ces solutions, laquelle est la plus légère ?"
                ),
            }
        ]
    },
    config=config,
)

print(result["messages"][-1].content)

Selon les informations disponibles, Leaflet est considéré comme l'une des bibliothèques JavaScript les plus légères pour l'affichage de cartes interactives. Il est souvent utilisé pour les applications web qui nécessitent une performance optimale et une petite taille de code.

Voici quelques caractéristiques de Leaflet qui en font une solution légère :

* Taille de code : environ 30 Ko
* Fonctionnalités : affichage de cartes interactives, ajout de couches, traçabilité, etc.
* Licence : open source

Cependant, il est important de noter que la légèreté d'une bibliothèque dépend également de la version et des fonctionnalités spécifiques que l'on utilise. Il est donc recommandé de vérifier les caractéristiques de la version que vous souhaitez utiliser.

Si vous cherchez une solution encore plus légère, vous pouvez également considérer d'autres bibliothèques comme OpenLayers, qui est également une option populaire pour l'affichage de cartes 2-D.


#### 6.bis Invocation en chunk

In [59]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

config_new = {
    "configurable": {
        "thread_id": "projet-design"
    }
}

for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Type de projet : site web de création d'affiche avec des assets qu'on upload (png,jpg,icon)\n"
                    "Stack : site web Angular"
                ),
            }
        ]
    },
    stream_mode="values",
    config=config_new,
):

    latest = chunk["messages"][-1]

    if isinstance(latest, HumanMessage):
        print(f"\n Utilisateur :\n{latest.content}")

    elif isinstance(latest, AIMessage):
        if latest.tool_calls:
            print("\n L'agent décide d'utiliser les outils :")
            for tool_call in latest.tool_calls:
                print(f"  • {tool_call['name']}")
                print(f"    Arguments : {tool_call['args']}")
        elif latest.content:
            print(f"\n Réponse finale :\n{latest.content}")

    elif isinstance(latest, ToolMessage):
        print("\n Résultat de Tavily :")
        print(latest.content)


 Utilisateur :
Type de projet : site web de création d'affiche avec des assets qu'on upload (png,jpg,icon)
Stack : site web Angular

 L'agent décide d'utiliser les outils :
  • tavily_search
    Arguments : {'query': 'bibliothèque image upload Angular site web création affiche'}
  • tavily_search
    Arguments : {'query': 'bibliothèque image traitement Angular site web création affiche'}
  • tavily_search
    Arguments : {'query': 'bibliothèque image compression Angular site web création affiche'}

 Résultat de Tavily :
{"query": "bibliothèque image compression Angular site web création affiche", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://stackoverflow.com/questions/70415693/can-you-compress-angular-image-assets-on-build", "title": "Can you compress angular image assets on build?", "content": "What I want I have very big images in my assets, which slows down the site by a lot for slower networks. (you can read more about the topic on this l

In [60]:
def demander_agent(
    user_input: str,
    thread_id: str = "conversation-1",
) -> str:
    """
    Envoie une requête à l'agent et retourne uniquement sa réponse finale.

    Parameters
    ----------
    user_input : str
        Question ou besoin de l'utilisateur.
    thread_id : str
        Identifiant de la conversation utilisé par le checkpointer.

    Returns
    -------
    str
        Réponse finale générée par l'agent.
    """

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input,
                }
            ]
        },
        config=config,
    )

    final_response = result["messages"][-1].content

    print(final_response)

    return final_response

### 7. Utilisation

In [61]:
demander_agent(
    user_input=(
        "Type de projet : site web de cartographie 2D interactive\n"
        "Stack : JavaScript"
    ),
    thread_id="projet-cartographie-bis",
)

| Solution | Description | Licence ou tarification |
| --- | --- | --- |
| ArcGIS Maps SDK for JavaScript | Bibliothèque utilisée pour créer des applications cartographiques Web captivantes avec ArcGIS Maps SDK for JavaScript, Calcite Design System et des bibliothèques cartographiques Open Source. | À vérifier |
| Leaflet | Bibliothèque open-source JavaScript pour créer des cartes interactives et mobile-friendly | Open source |
| CARTO.js | Bibliothèque open-source JavaScript pour créer des cartes interactives et intégrer avec les API du CARTO Engine | Open source |


'| Solution | Description | Licence ou tarification |\n| --- | --- | --- |\n| ArcGIS Maps SDK for JavaScript | Bibliothèque utilisée pour créer des applications cartographiques Web captivantes avec ArcGIS Maps SDK for JavaScript, Calcite Design System et des bibliothèques cartographiques Open Source. | À vérifier |\n| Leaflet | Bibliothèque open-source JavaScript pour créer des cartes interactives et mobile-friendly | Open source |\n| CARTO.js | Bibliothèque open-source JavaScript pour créer des cartes interactives et intégrer avec les API du CARTO Engine | Open source |'

| Solution | Description | Licence ou tarification |
| --- | --- | --- |
| ArcGIS Maps SDK for JavaScript | Bibliothèque utilisée pour créer des applications cartographiques Web captivantes avec ArcGIS Maps SDK for JavaScript, Calcite Design System et des bibliothèques cartographiques Open Source. | À vérifier |
| Leaflet | Bibliothèque open-source JavaScript pour créer des cartes interactives et mobile-friendly | Open source |
| CARTO.js | Bibliothèque open-source JavaScript pour créer des cartes interactives et intégrer avec les API du CARTO Engine | Open source |
| Solution | Description | Licence ou tarification |\n| --- | --- | --- |\n| ArcGIS Maps SDK for JavaScript | Bibliothèque utilisée pour créer des applications cartographiques Web captivantes avec ArcGIS Maps SDK for JavaScript, Calcite Design System et des bibliothèques cartographiques Open Source. | À vérifier |\n| Leaflet | Bibliothèque open-source JavaScript pour créer des cartes interactives et mobile-friendly | Open source |\n| CARTO.js | Bibliothèque open-source JavaScript pour créer des cartes interactives et intégrer avec les API du CARTO Engine | Open source |

# Aides

La section suivante contient des aides pour d'autres fonctionnalités

#### invocation directe de tavily

In [45]:
search.invoke({"query": "Que s'est-il passé au dernier tournoi de tennis de Roland Garros ?"})

{'query': "Que s'est-il passé au dernier tournoi de tennis de Roland Garros ?",
 'follow_up_questions': None,
 'answer': 'Le dernier tournoi de Roland-Garros a vu Rafael Nadal remporter son 14e titre. La finale masculine a été gagnée par Novak Djokovic. La finale féminine a été gagnée par Iga Świątek.',
 'images': [],
 'results': [{'url': 'https://lespetitscitoyens.com/le_journal/le-tournoi-de-roland-garros-a-pris-fin',
   'title': 'Le tournoi de Roland-Garros a pris fin !',
   'content': "Le record du plus grand nombre de victoires à Roland-Garros est détenu par le joueur espagnol Rafael Nadal : il a gagné treize fois ! Chez les femmes, c'est l'",
   'score': 0.5835125,
   'raw_content': None},
  {'url': 'https://fr.wikipedia.org/wiki/Internationaux_de_France_de_tennis',
   'title': 'Internationaux de France de tennis — Wikipédia',
   'content': "- 1.2.3 Avant-guerre. + 1.3 De la Seconde Guerre mondiale au professionnalisme. - 1.3.1 Tournoi pendant la guerre. - 1.3.2 Après-guerre et v

#### Invocation avec ToolCall

In [47]:
model_generated_tool_call = {
    "args": {"query": "pays hôte de l'euro 2024"},
    "id": "1",
    "name": "tavily",
    "type": "tool_call",
}

In [48]:
tool_msg = search.invoke(model_generated_tool_call)

In [49]:
# Le contenu est une chaîne JSON de résultats
print(tool_msg.content[:400])

{"query": "pays hôte de l'euro 2024", "follow_up_questions": null, "answer": "L'UEFA EURO 2024 se déroulera en Allemagne, avec des matchs dans dix stades de classe mondiale, dont le stade Olympique de Berlin et le Volksparkstadion de Hambourg. La finale se jouera le 14 juillet 2024 à Berlin.", "images": [], "results": [{"url": "https://www.20minutes.fr/sport/football/2344335-20180927-euro-2024-all


In [50]:
import json

# L'artefact est un dict avec des résultats bruts plus riches
# Si tool_msg.artifact est None, on utilise tool_msg.content (qui est une chaîne JSON)
data_source = tool_msg.artifact if tool_msg.artifact is not None else json.loads(tool_msg.content)
{k: type(v) for k, v in data_source.items()}

{'query': str,
 'follow_up_questions': NoneType,
 'answer': str,
 'images': list,
 'results': list,
 'response_time': float,
 'request_id': str}

In [51]:
# Abréger les résultats à des fins de démonstration
print(json.dumps({k: str(v)[:200] for k, v in data_source.items()}, indent=2))

{
  "query": "pays h\u00f4te de l'euro 2024",
  "follow_up_questions": "None",
  "answer": "L'UEFA EURO 2024 se d\u00e9roulera en Allemagne, avec des matchs dans dix stades de classe mondiale, dont le stade Olympique de Berlin et le Volksparkstadion de Hambourg. La finale se jouera le 14 juillet ",
  "images": "[]",
  "results": "[{'url': 'https://www.20minutes.fr/sport/football/2344335-20180927-euro-2024-allemagne-designee-pays-hote-competition-turquie-devra-encore-patienter', 'title': \"Euro 2024: L'Allemagne est d\u00e9sign\u00e9e pay",
  "response_time": "1.8",
  "request_id": "454c4e8e-4699-412c-be3f-2044281acd63"
}


## Utilisation des modèles de langage

### MistraAI

In [ ]:
%pip install langchain-mistralai

In [ ]:
import os
os.environ["MISTRAL_API_KEY"] = "Ta clé API Mistral"

from langchain_mistralai import ChatMistralAI
model = ChatMistralAI(model="mistral-large-latest")

### OpenAI

In [ ]:
%pip install langchain-openai

In [ ]:
import os
os.environ["OPENAI_API_KEY"] ="Ta clé API OpenAI"

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content="salut !")])
response.content

In [ ]:
model_with_tools = model.bind_tools(tools)

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="Salut !")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="Quel temps fait-il à Lisbonne ?")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")